# Z24 PDT: 1DCNN-LSTM-ResNet Training

**Author:** TRAN VAN PHUONG

This notebook trains a 17-class condition classifier from the CSV files produced by `PDT_cleaning.ipynb`. The model architecture and training settings follow the teacher's `DCNN-LSTM-ResNet.py`; only the data input differs.

## Experiment design

- Input: a 10-second window sampled at 100 Hz, giving 1,000 time samples.
- Channels: the five sensors shared by every PDT setup: R1V, R2L, R2T, R2V, and R3V.
- Output: one of 17 condition labels (0--16), corresponding to condition folders 1--17.
- Split: six recordings per condition for training, one for validation, and two for testing.
- Isolation: all CSV segments from one source MAT recording stay in the same split.
- Normalization: mean and standard deviation are calculated from training data only.

AVT and FVT are trained separately. Change `MEASUREMENT` below to run the other measurement type.

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import os
import sys

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

start = Path.cwd().resolve()
ROOT = next(
    (path for path in (start, *start.parents)
     if (path / "src" / "data" / "pdt_training_data.py").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from the shm project or its notebooks folder")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix

from src.models.dcnn_lstm_resnet import build_model
from src.data.pdt_training_data import (
    common_channels,
    latest_complete_run,
    load_windows,
    normalize_from_train,
    read_manifest,
    split_by_recording,
)

print("Project root:", ROOT)
print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

## Configuration

The default AVT run is a complete experiment. A training run can take a long time on CPU. Following the teacher's source, early stopping monitors validation accuracy with a patience of 30 epochs.

In [ ]:
MEASUREMENT = "avt"       # Change to "fvt" for a separate FVT experiment
WINDOW_SAMPLES = 1000       # 10 seconds at 100 Hz
BATCH_SIZE = 64
EPOCHS = 100
SEED = 42

tf.keras.utils.set_random_seed(SEED)

RUN_DIR = latest_complete_run(ROOT, MEASUREMENT)
MANIFEST = read_manifest(RUN_DIR)
CHANNELS = list(common_channels(MANIFEST))
NUM_CLASSES = len({int(row["label"]) for row in MANIFEST})

print("Clean-data run:", RUN_DIR)
print("Manifest rows:", len(MANIFEST))
print("Common channels:", CHANNELS)
print("Classes:", NUM_CLASSES)

assert NUM_CLASSES == 17, "Expected all 17 PDT conditions"
assert CHANNELS == ["R1V", "R2L", "R2T", "R2V", "R3V"], (
    "The common sensor layout differs from the verified PDT layout", CHANNELS
)

## Recording-level split

The split is stratified by condition. It uses the source MAT recording as the group, so neighboring segments from a recording cannot leak into another split.

In [ ]:
SPLITS = split_by_recording(MANIFEST, seed=SEED)

split_summary = []
for split_name, rows in SPLITS.items():
    split_summary.append({
        "split": split_name,
        "recordings": len({row["source"] for row in rows}),
        "csv_segments": len(rows),
        "complete_windows": sum(int(row["samples"]) // WINDOW_SAMPLES for row in rows),
        "conditions": len({int(row["label"]) for row in rows}),
    })

split_summary = pd.DataFrame(split_summary).set_index("split")
display(split_summary)

source_sets = {
    name: {row["source"] for row in rows}
    for name, rows in SPLITS.items()
}
assert source_sets["train"].isdisjoint(source_sets["validation"])
assert source_sets["train"].isdisjoint(source_sets["test"])
assert source_sets["validation"].isdisjoint(source_sets["test"])
print("Recording isolation: OK")

## Load fixed-length windows

Each 6,000-row CSV produces six non-overlapping windows. The final short CSV in a recording contributes every complete 1,000-row window; any remainder shorter than 1,000 rows is discarded. Arrays use the Keras layout `(examples, time samples, channels)`.

In [ ]:
x_train, y_train = load_windows(RUN_DIR, SPLITS["train"], CHANNELS, WINDOW_SAMPLES)
x_validation, y_validation = load_windows(
    RUN_DIR, SPLITS["validation"], CHANNELS, WINDOW_SAMPLES
)
x_test, y_test = load_windows(RUN_DIR, SPLITS["test"], CHANNELS, WINDOW_SAMPLES)

for name, x, y in (
    ("train", x_train, y_train),
    ("validation", x_validation, y_validation),
    ("test", x_test, y_test),
):
    print(f"{name:10s} x={x.shape}, y={y.shape}, memory={x.nbytes / 1024**2:.1f} MB")
    assert x.shape[1:] == (WINDOW_SAMPLES, len(CHANNELS))
    assert np.isfinite(x).all()
    assert set(np.unique(y)) == set(range(NUM_CLASSES))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for split_name, labels in (
    ("train", y_train),
    ("validation", y_validation),
    ("test", y_test),
):
    counts = np.bincount(labels, minlength=NUM_CLASSES)
    axes[0].plot(np.arange(1, NUM_CLASSES + 1), counts, marker="o", label=split_name)
axes[0].set_title("Windows per condition and split")
axes[0].set_xlabel("Condition")
axes[0].set_ylabel("Windows")
axes[0].set_xticks(range(1, NUM_CLASSES + 1))
axes[0].grid(alpha=0.3)
axes[0].legend()

time_seconds = np.arange(WINDOW_SAMPLES) / 100.0
for channel_index, channel_name in enumerate(CHANNELS):
    axes[1].plot(time_seconds, x_train[0, :, channel_index], label=channel_name)
axes[1].set_title("One raw training window")
axes[1].set_xlabel("Time (seconds)")
axes[1].set_ylabel("Recorded value")
axes[1].grid(alpha=0.3)
axes[1].legend()
plt.tight_layout()
plt.show()

## Normalize and create batches

The validation and test arrays use the training mean and standard deviation. Their own statistics are never used to fit preprocessing.

In [ ]:
x_train, x_validation, x_test, channel_mean, channel_std = normalize_from_train(
    x_train, x_validation, x_test
)

normalization = pd.DataFrame({
    "channel": CHANNELS,
    "training_mean": channel_mean,
    "training_std": channel_std,
})
display(normalization)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(len(x_train), seed=SEED, reshuffle_each_iteration=True)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
validation_ds = (
    tf.data.Dataset.from_tensor_slices((x_validation, y_validation))
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
test_ds = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

## Build the model

The input shape is ordered as time samples first and channels second, which is the convention expected by Keras Conv1D and LSTM layers.

In [ ]:
tf.keras.backend.clear_session()
model = build_model(
    input_shape=(WINDOW_SAMPLES, len(CHANNELS)),
    num_classes=NUM_CLASSES,
)
model.summary()

In [ ]:
timestamp = datetime.now().strftime("%d-%m-%Y_%H-%M-%S")
ARTIFACT_DIR = ROOT / "artifacts" / "dcnn_lstm_resnet" / f"{MEASUREMENT}_{timestamp}"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=False)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=30, restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        ARTIFACT_DIR / "model_1DCNN_LSTM_ResNet.h5",
        save_best_only=True,
        monitor="val_accuracy",
        mode="max",
    ),
]
print("Artifacts:", ARTIFACT_DIR)

## Train

Validation accuracy guides early stopping and model checkpointing, following the teacher's source. Test data is not passed to `fit()`.

In [ ]:
history = model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

In [ ]:
history_frame = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
history_frame[["loss", "val_loss"]].plot(ax=axes[0])
axes[0].set_title("Training and validation loss")
axes[0].set_xlabel("Epoch")
axes[0].grid(alpha=0.3)
history_frame[["accuracy", "val_accuracy"]].plot(ax=axes[1])
axes[1].set_title("Training and validation accuracy")
axes[1].set_xlabel("Epoch")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Final test evaluation

Run this cell after training has finished. The test split provides the final unbiased estimate for this experiment.

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
probabilities = model.predict(test_ds, verbose=0)
predictions = probabilities.argmax(axis=1)

condition_names = [
    next(row["condition_name"] for row in MANIFEST if int(row["label"]) == label)
    for label in range(NUM_CLASSES)
]
report = classification_report(
    y_test,
    predictions,
    labels=np.arange(NUM_CLASSES),
    target_names=[f"{index + 1:02d}: {name}" for index, name in enumerate(condition_names)],
    output_dict=True,
    zero_division=0,
)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
display(pd.DataFrame(report).T.round(3))

matrix = confusion_matrix(y_test, predictions, labels=np.arange(NUM_CLASSES))
fig, ax = plt.subplots(figsize=(12, 12))
ConfusionMatrixDisplay(
    confusion_matrix=matrix,
    display_labels=np.arange(1, NUM_CLASSES + 1),
).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"{MEASUREMENT.upper()} test confusion matrix")
ax.set_xlabel("Predicted condition")
ax.set_ylabel("True condition")
plt.tight_layout()
plt.show()

## Save reproducible outputs

The saved model must be used together with the same channel order, window length, and normalization values.

In [ ]:
model.save(ARTIFACT_DIR / "final_model.keras")
history_frame.to_csv(ARTIFACT_DIR / "history.csv", index_label="epoch")
pd.DataFrame(report).T.to_csv(ARTIFACT_DIR / "classification_report.csv")
np.savetxt(ARTIFACT_DIR / "confusion_matrix.csv", matrix, fmt="%d", delimiter=",")

experiment = {
    "measurement": MEASUREMENT,
    "clean_data_run": str(RUN_DIR.relative_to(ROOT)),
    "window_samples": WINDOW_SAMPLES,
    "sampling_rate_hz": 100,
    "channels": CHANNELS,
    "num_classes": NUM_CLASSES,
    "batch_size": BATCH_SIZE,
    "maximum_epochs": EPOCHS,
    "completed_epochs": len(history.history["loss"]),
    "seed": SEED,
    "recordings_per_split": {
        name: len({row["source"] for row in rows})
        for name, rows in SPLITS.items()
    },
    "normalization_mean": channel_mean.tolist(),
    "normalization_std": channel_std.tolist(),
    "test_loss": float(test_loss),
    "test_accuracy": float(test_accuracy),
}
(ARTIFACT_DIR / "experiment.json").write_text(
    json.dumps(experiment, indent=2), encoding="utf-8"
)
print("Saved model and evaluation files to:", ARTIFACT_DIR)